# 02 — Análise estatística das amostras e classificação unitária (MaxVer)

**Executar uma vez por imagem SAR**, depois do notebook 01.

| Etapa | Conteúdo | Dissertação |
|---|---|---|
| 1 | amostras de treino (notebook 01) → valores da imagem | 4.3.1.1 |
| 2 | generalização / balanceamento / remoção de duplicados (opcionais) | — |
| 3 | estatística descritiva (intensidade e amplitude) | — |
| 4 | aderência: Gama, Normal, Mardia; ENL iterativo | 4.3.1.2 |
| 5 | separabilidade e dendrograma (opcional) | 4.3.1.3 |
| 6 | validação cruzada de cada modelo (70/30, N iterações, agregação na matriz) | 4.3.1.4 |
| 7 | modelo representante (3º quartil) e matriz de confusão | 4.3.1.4 |
| 8 | imagens de verossimilhança e classificação MaxVer | 4.3.1.4 |

**Gravação:** três arquivos de resultados, mais uma imagem de verossimilhança por modelo, que é necessária ao CMAP. Os scripts R gravavam cerca de 40 arquivos por imagem.

| Arquivo | Conteúdo |
|---|---|
| `02_modelagem.xlsx` | todas as tabelas, uma aba por tabela |
| `modelos.json` | modelos representantes + ENL + sementes + parâmetros usados |
| `maxver_modelos.tif` | uma banda por modelo |
| `verossimilhanca/<modelo>.tif` | log-verossimilhança, uma banda por classe (nome da classe na banda) |

## Parâmetros

`MODELOS` define as modelagens. Os campos são:

- `atributos`: nomes das polarizações;
- `transformacao`: `None` (intensidade) ou `'sqrt'` (amplitude);
- `enl`: `'HH'`, `'HV'`, `'media'` (média dos dois), um número ou `None`.

Os seis modelos abaixo são os da dissertação (Experimento 1). Os gaussianos em intensidade, usados em `analise_2_5classes.R`, estão comentados.

In [ ]:
CONFIG = 'config_projeto.json'
DATA = '2010'

MODELOS = {
    'HH_gama':         {'distribuicao': 'gamma',    'atributos': ['HH'],       'transformacao': None,   'enl': 'HH'},
    'HV_gama':         {'distribuicao': 'gamma',    'atributos': ['HV'],       'transformacao': None,   'enl': 'HV'},
    'par_intensidade': {'distribuicao': 'intensity_joint_distribution',
                                                    'atributos': ['HH', 'HV'], 'transformacao': None,   'enl': 'media'},
    'HH_gauss_A':      {'distribuicao': 'gaussian', 'atributos': ['HH'],       'transformacao': 'sqrt', 'enl': None},
    'HV_gauss_A':      {'distribuicao': 'gaussian', 'atributos': ['HV'],       'transformacao': 'sqrt', 'enl': None},
    'gauss_biv_A':     {'distribuicao': 'gaussian', 'atributos': ['HH', 'HV'], 'transformacao': 'sqrt', 'enl': None},
    # 'HH_gauss_I':    {'distribuicao': 'gaussian', 'atributos': ['HH'],       'transformacao': None,   'enl': None},
    # 'HV_gauss_I':    {'distribuicao': 'gaussian', 'atributos': ['HV'],       'transformacao': None,   'enl': None},
    # 'gauss_biv_I':   {'distribuicao': 'gaussian', 'atributos': ['HH', 'HV'], 'transformacao': None,   'enl': None},
}

N_ITERACOES = 1000               # validação cruzada (dissertação: 1000)
PROP_TREINO = 0.7
NIVEL_AGREGACAO = 'lvl_2'        # agregação NA MATRIZ de confusão (None = sem agregação)
NIVEL_GENERALIZACAO = None       # agregação NAS AMOSTRAS antes da modelagem (ex.: 'lvl_2')
RAZAO_DOWNSAMPLE = None          # ex.: 5 (downsample_data); None = sem balanceamento
SEMENTE_DOWNSAMPLE = 1234
REMOVER_DUPLICADOS = True        # filter(!duplicated(HH) & !duplicated(HV)) por classe
CRITERIO_REPRESENTANTE = 'overall_accuracy'   # 3º quartil da 'overall_accuracy' (texto) ou 'f1_macro' (script 13)
NORMALIZAR_METRICAS = True       # True = acurácia balanceada (código original); False = convencional
EXECUTAR_SEPARABILIDADE = True
SALVAR_FIGURAS = False

## Configuração e arquivos

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))      # pasta que contém sar_classification

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from sar_classification import (autocorrelation as ac, sampling, samples as smp, fitting as fit,
                                separability as sep, models as mdl, validation as val, raster_io as rio,
                                transitions as trn, cmap, trajectory_assessment as ta, plots, reporting)

with open(CONFIG, encoding='utf-8') as f:
    cfg = json.load(f)
RESULTADOS = Path(cfg['raiz_resultados'])
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

img = cfg['imagens'][DATA]
BANDAS = img['bandas']
NIVEIS = cfg['niveis_hierarquicos']
AGREGACAO = NIVEIS[NIVEL_AGREGACAO] if NIVEL_AGREGACAO else None
LEGENDA_FINAL = cfg.get('legenda_final') or {}
OUT = RESULTADOS / DATA
(OUT / 'verossimilhanca').mkdir(parents=True, exist_ok=True)
ARQ_PLANILHA, ARQ_MODELOS = OUT / '02_modelagem.xlsx', OUT / 'modelos.json'
pdf = reporting.open_pdf(OUT / 'figuras_02.pdf') if SALVAR_FIGURAS else None
tabelas = {}                                              # todas as tabelas -> 02_modelagem.xlsx

## 1. Amostras de treinamento

In [ ]:
treino = gpd.read_file(OUT / 'amostras.gpkg', layer='treino')
df = sampling.extract_samples(treino, img['imagem'], band_names=BANDAS)[['Class', 'HH', 'HV']]
print(f'{len(df)} amostras | classes: {df["Class"].value_counts().to_dict()}')

## 2. Generalização, balanceamento e duplicados (opcionais)

In [ ]:
if NIVEL_GENERALIZACAO:
    df = smp.generalize_classes(df, [NIVEIS[NIVEL_GENERALIZACAO]], level=1)
if RAZAO_DOWNSAMPLE:
    df = smp.downsample_samples(df, ratio=RAZAO_DOWNSAMPLE, random_state=SEMENTE_DOWNSAMPLE)
if REMOVER_DUPLICADOS:
    df = smp.drop_duplicates_by_class(df, ['HH', 'HV'])
CLASSES = [c for c in cfg['classes'] if c in set(df['Class'])] + sorted(set(df['Class']) - set(cfg['classes']))
df_a = df.assign(HH=np.sqrt(df['HH']), HV=np.sqrt(df['HV']))         # amplitude (só para estatística/aderência)
print('Classes modeladas:', CLASSES)

## 3. Estatística descritiva

In [ ]:
tabelas['estatisticas'] = pd.concat([smp.describe_by_class(df).assign(Formato='intensidade'),
                                     smp.describe_by_class(df_a).assign(Formato='amplitude')], ignore_index=True)
tabelas['estatisticas'].round(5)

In [ ]:
plots.boxplots_by_class(df, CLASSES, title=f'Intensidade — {DATA}', pdf=pdf)

## 4. Aderência e ENL

| Teste | Dados |
|---|---|
| Gama (α, β livres, MV) + KS | intensidade, por polarização |
| Normal (MV) + KS | intensidade e amplitude, por polarização |
| Mardia | intensidade e amplitude, bivariado |
| ENL iterativo | por polarização; o par de intensidades usa a média |


In [ ]:
aderencia = []
for b in ['HH', 'HV']:
    aderencia.append(fit.gamma_fitting(df[['Class', b]], CLASSES, verbose=False).assign(Polarizacao=b, Formato='intensidade', Distribuicao='Gama'))
    aderencia.append(fit.normal_fitting(df[['Class', b]], CLASSES, verbose=False).assign(Polarizacao=b, Formato='intensidade', Distribuicao='Normal'))
    aderencia.append(fit.normal_fitting(df_a[['Class', b]], CLASSES, verbose=False).assign(Polarizacao=b, Formato='amplitude', Distribuicao='Normal'))
tabelas['aderencia_univariada'] = pd.concat(aderencia, ignore_index=True)
tabelas['mardia'] = pd.concat([fit.normal_fitting_mv(df, CLASSES).assign(Formato='intensidade'),
                               fit.normal_fitting_mv(df_a, CLASSES).assign(Formato='amplitude')], ignore_index=True)
pd.crosstab([tabelas['aderencia_univariada'].Distribuicao, tabelas['aderencia_univariada'].Formato,
             tabelas['aderencia_univariada'].Polarizacao], tabelas['aderencia_univariada'].H0)   # resumo (Tabelas 5.3 e 5.4)

In [ ]:
ENL, enl_tabs = {}, []
for b in ['HH', 'HV']:
    r = fit.iterative_gamma_fitting(df[['Class', b]], CLASSES, verbose=False)
    if r is None:
        raise ValueError(f'ENL de {b} não determinado (nenhuma classe aderiu à Gama).')
    ENL[b] = r['best_enl']
    enl_tabs.append(r['fitting_result'].assign(Polarizacao=b))
    tabelas[f'enl_candidatos_{b}'] = r['result_counting']
ENL['media'] = 0.5 * (ENL['HH'] + ENL['HV'])
tabelas['enl_ajuste_final'] = pd.concat(enl_tabs, ignore_index=True)                  # Tabela 5.5
tabelas['enl'] = pd.DataFrame({'chave': list(ENL), 'ENL': list(ENL.values())})
print({k: round(v, 4) for k, v in ENL.items()})

## 5. Separabilidade (opcional)

Bhattacharyya da Gama em cada polarização, com o ENL estimado, e dendrograma de Ward, como apoio à definição da legenda (Seção 4.3.1.3 da dissertação).

In [ ]:
if EXECUTAR_SEPARABILIDADE:
    for b in ['HH', 'HV']:
        B = sep.bhattacharyya_matrix(df[['Class', b]], CLASSES, 'gamma', ENL[b])
        tabelas[f'bhattacharyya_{b}'] = B.round(6).reset_index().rename(columns={'index': 'Classe'})
        Z = sep.hierarchical_clustering(B)
        plots.plot_dendrogram(Z, CLASSES, title=f'Bhattacharyya (Gama) {b} — {DATA}', pdf=pdf)
        print(b, sep.extract_hierarchy(Z, CLASSES))

## 6. Validação cruzada de cada modelo

São 70/30 estratificados, com `N_ITERACOES` repetições e agregação `NIVEL_AGREGACAO` na matriz de confusão. As sementes são 1..N, o que torna o processo reprodutível.

In [ ]:
def enl_do_modelo(spec):
    e = spec['enl']
    return None if e is None else (ENL[e] if isinstance(e, str) else float(e))

cv = {}
for nome, spec in MODELOS.items():
    print(f'--- {nome}')
    cv[nome] = val.cross_validation(df[['Class'] + spec['atributos']], CLASSES, spec['distribuicao'],
                                    enl_do_modelo(spec), n_iterations=N_ITERACOES, train_proportion=PROP_TREINO,
                                    aggregation=AGREGACAO, normalize=NORMALIZAR_METRICAS,
                                    rank_by=CRITERIO_REPRESENTANTE, transform=spec['transformacao'], verbose=False)
    print(f"    OA média = {cv[nome]['overall_accuracy'].mean():.4f} | F1 macro médio = {cv[nome]['f1_macro'].mean():.4f}")

tabelas['validacao_global'], tabelas['validacao_classes'] = val.summarize_models(cv)
tabelas['validacao_iteracoes'] = val.iterations_table(cv)
tabelas['validacao_global'].round(4)

In [ ]:
plots.plot_model_comparison(tabelas['validacao_global'], 'OA', title=f'Acurácia Global — MaxVer {DATA}', pdf=pdf)
plots.plot_model_comparison(tabelas['validacao_global'], 'F1', title=f'F1 macro — MaxVer {DATA}', pdf=pdf)
for metrica in ['PA', 'UA']:
    plots.plot_class_metrics(tabelas['validacao_classes'], metrica, title=f'{metrica} por classe — {DATA}', pdf=pdf)

## 7. Modelo representante de cada modelagem

O representante é a iteração cuja métrica `CRITERIO_REPRESENTANTE` está no **3º quartil**. A iteração é refeita exatamente com `reproduce_iteration`.

In [ ]:
modelos, matrizes, representantes = {}, [], []
for nome, spec in MODELOS.items():
    sel = cv[nome]['selected']
    rep = val.reproduce_iteration(df[['Class'] + spec['atributos']], CLASSES, sel['third_quartile_seed'],
                                  spec['distribuicao'], enl_do_modelo(spec), train_proportion=PROP_TREINO,
                                  aggregation=AGREGACAO, normalize=NORMALIZAR_METRICAS, transform=spec['transformacao'])
    modelos[nome] = rep['model']
    representantes.append({'Modelo': nome, 'criterio': sel['rank_by'], 'semente_3q': sel['third_quartile_seed'],
                           'semente_mediana': sel['median_seed'], 'semente_melhor': sel['best_seed'],
                           'OA': rep['overall_accuracy'], 'F1_macro': rep['f1_macro'], 'ENL': enl_do_modelo(spec)})
    cm = rep['confusion_matrix'].rename(index=LEGENDA_FINAL, columns=LEGENDA_FINAL)
    matrizes.append(cm.stack().rename('n').reset_index().assign(Modelo=nome))
    plots.bubble_confusion_plot(cm, title=f'{nome} — 3º quartil ({DATA})', pdf=pdf)
tabelas['representantes'] = pd.DataFrame(representantes)
tabelas['matrizes_3q'] = pd.concat(matrizes, ignore_index=True)[['Modelo', 'Prediction', 'Reference', 'n']]
tabelas['representantes'].round(4)

## 8. Imagens de verossimilhança e classificação MaxVer

- **Verossimilhança:** um arquivo por modelo, com uma banda por classe e o nome da classe gravado na banda. É gravada em **log**, imune a *underflow*. O CMAP (notebook 03) lê esse formato com `tipo = 'log'`.
- **Classificação:** todos os modelos num único GeoTIFF, com uma banda por modelo e códigos de classe na ordem de `cfg['classes']`.

A transformação (amplitude) é aplicada automaticamente, porque fica gravada em cada modelo.

In [ ]:
codigos = {c: i + 1 for i, c in enumerate(CLASSES)}
arquivos_verossimilhanca = {}
for nome, modelo in modelos.items():
    arquivos_verossimilhanca[nome] = rio.likelihood_raster(img['imagem'], modelo, OUT / 'verossimilhanca' / f'{nome}.tif',
                                                           band_names=BANDAS, log=True)
arq_maxver = rio.classification_stack(img['imagem'], modelos, OUT / 'maxver_modelos.tif', codigos, band_names=BANDAS)
tabelas['legenda_maxver'] = pd.DataFrame({'Classe': list(codigos), 'ID': list(codigos.values())})
print('Verossimilhanças:', *arquivos_verossimilhanca.values(), sep='\n  ')
print('Classificações MaxVer (1 banda por modelo):', arq_maxver)

In [ ]:
import rasterio
with rasterio.open(arq_maxver) as s:
    fig, axes = plt.subplots(2, (s.count + 1) // 2, figsize=(4.2 * ((s.count + 1) // 2), 7.5), squeeze=False)
    for ax, b in zip(axes.ravel(), range(1, s.count + 1)):
        ax.imshow(np.ma.masked_equal(s.read(b), 0), cmap='tab10', vmin=1, vmax=10, interpolation='nearest')
        ax.set_title(s.descriptions[b - 1]); ax.axis('off')
    for ax in axes.ravel()[s.count:]:
        ax.axis('off')
plt.suptitle(f'Classificação MaxVer por modelo — {DATA}'); plt.tight_layout()
if pdf: pdf.savefig(fig)
plt.show()

## 9. Gravação

In [ ]:
tabelas['parametros'] = pd.DataFrame({'parametro': ['DATA', 'N_ITERACOES', 'PROP_TREINO', 'NIVEL_AGREGACAO', 'NIVEL_GENERALIZACAO',
                                                     'RAZAO_DOWNSAMPLE', 'REMOVER_DUPLICADOS', 'CRITERIO_REPRESENTANTE',
                                                     'NORMALIZAR_METRICAS', 'BANDAS', 'MODELOS'],
                                       'valor': [DATA, N_ITERACOES, PROP_TREINO, NIVEL_AGREGACAO, NIVEL_GENERALIZACAO,
                                                 RAZAO_DOWNSAMPLE, REMOVER_DUPLICADOS, CRITERIO_REPRESENTANTE,
                                                 NORMALIZAR_METRICAS, str(BANDAS), json.dumps(MODELOS)]})
reporting.save_workbook(ARQ_PLANILHA, tabelas)
mdl.save_models(modelos, ARQ_MODELOS, extra={'data': DATA, 'ENL': ENL, 'codigos': codigos,
                                             'representantes': tabelas['representantes'].to_dict('records')})
if pdf: pdf.close()
print('Arquivos desta etapa:', *sorted(p.name for p in OUT.iterdir()), sep='\n  ')